In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

uniprot_path = ROOT / "data/raw/uniprot/uniprotkb_organism_id_9606_AND_reviewed_2026_10_12.tsv.gz"

uniprot_df = pd.read_csv(
    uniprot_path,
    sep="\t",
    compression="gzip",
    dtype=str
)

In [2]:
# Keep only proteins which have a valid STRING ID
protein_df = uniprot_df[
    uniprot_df["STRING"].notna()
    & uniprot_df["STRING"].str.strip().ne("")
].copy()

# Keep only columns needed in later graph/model steps
protein_df = protein_df[
    [
        "Entry",
        "Entry Name",
        "Gene Names (primary)",
        "Protein names",
        "Sequence",
        "Gene Ontology IDs",
        "Involvement in disease",
        "STRING",
    ]
].rename(
    columns={
        "Entry": "uniprot_id",
        "Entry Name": "entry_name",
        "Gene Names (primary)": "gene_name",
        "Protein names": "protein_name",
        "Sequence": "sequence",
        "Gene Ontology IDs": "go_ids",
        "Involvement in disease": "uniprot_disease_annotation",
        "STRING": "string_id",
    }
)

# Clean STRING IDs and add useful derived columns
protein_df["string_id"] = protein_df["string_id"].str.strip().str.rstrip(";")
protein_df["sequence_length"] = protein_df["sequence"].str.len().astype("int32")
protein_df["has_go_labels"] = protein_df["go_ids"].fillna("").str.strip().ne("")
protein_df["has_uniprot_disease_annotation"] = (
    protein_df["uniprot_disease_annotation"].fillna("").str.strip().ne("")
)

protein_df.head()

,uniprot_id,entry_name,gene_name,protein_name,sequence,go_ids,uniprot_disease_annotation,string_id,sequence_length,has_go_labels,has_uniprot_disease_annotation
1,A0A096LP01,SIM26_HUMAN,SMIM26,Small integral membrane protein 26,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,GO:0005739; GO:0005741; GO:0019901; GO:0044325,NaN,9606.ENSP00000485316,95,True,False
2,A0A0B4J2F0,PIOS1_HUMAN,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1),MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,GO:0005739; GO:0005741; GO:1900101,NaN,9606.ENSP00000484893,54,True,False
5,A0A0U1RRE5,NBDY_HUMAN,NBDY,Negative regulator of P-body association (P-bo...,MGDQPCASGRSTLPPGNAREAKPPKKRCLLAPRWDYPEGTPNGGST...,GO:0000932; GO:0000956; GO:0010607,NaN,9606.ENSP00000489583,68,True,False
9,A0AV02,S12A8_HUMAN,SLC12A8,Solute carrier family 12 member 8 (Cation-chlo...,MTQMSQVQELFHEAAQQDALAQPQPWWKTQLFMWEPVLFGTWDGVF...,GO:0015377; GO:0016324; GO:0071805,DISEASE: Note=SLC12A8 has been identified as a...,9606.ENSP00000377112,714,True,True
10,A0AV96,RBM47_HUMAN,RBM47,RNA-binding protein 47 (RNA-binding motif prot...,MTAEDSTAAMSSDSAAGSSAKVPEGVAGAPNEAALLALMERTGYSM...,GO:0000381; GO:0003723; GO:0003729; GO:0003730...,NaN,9606.ENSP00000295971,593,True,False


In [3]:
# Validation checks
assert protein_df["uniprot_id"].is_unique
assert protein_df["string_id"].is_unique
assert protein_df["sequence"].notna().all()
assert (protein_df["sequence_length"] > 0).all()

print("Clean protein table shape:", protein_df.shape)
print("\nMissing values:")
print(protein_df.isna().sum())

print("\nGO-label coverage:")
print(protein_df["has_go_labels"].value_counts())

print("\nDisease-text coverage:")
print(protein_df["has_uniprot_disease_annotation"].value_counts())

Clean protein table shape: (18841, 11)

Missing values:
uniprot_id                            0
entry_name                            0
gene_name                             0
protein_name                          0
sequence                              0
go_ids                              568
uniprot_disease_annotation        13556
string_id                             0
sequence_length                       0
has_go_labels                         0
has_uniprot_disease_annotation        0
dtype: int64

GO-label coverage:
has_go_labels
True     18273
False      568
Name: count, dtype: int64

Disease-text coverage:
has_uniprot_disease_annotation
False    13556
True      5285
Name: count, dtype: int64


In [4]:
processed_dir = ROOT / "data/processed"
processed_dir.mkdir(exist_ok=True)

output_path = processed_dir / "human_proteins_with_string.parquet"

protein_df.to_parquet(output_path, index=False)

print("Saved:", output_path)
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")

Saved: c:\Users\rajdeep\Desktop\Capstone Project_GNN\data\processed\human_proteins_with_string.parquet
File size: 15.03 MB
